Sitadel

In [ ]:
%run Sitadel2_traitement_1auto.py

In [ ]:
temp=df1000[df1000['adresse'].str.contains(r'(parc|\bzi|\bzac|\bza)', case=False, na=False)]
print(f'{len(temp)}/{len(df1000)} labels "zone" dans les adresses')
df1000[
    df1000['adresse'].str.contains(r'(parc|\bzi|\bzac|\bza)', case=False, na=False) &  # Recherche sans \b pour trouver "parc" dans "LOGICPARC"
    df1000['adresse_BAN'].str.contains(r'(parc d|zi|zac|za|zone)', case=False, na=False)
][["adresse","adresse_BAN","score_BAN","SURF_ENT_CREEE","SURF_COM_CREEE",
     "SURF_IND_CREEE","DATE_REELLE_AUTORISATION","DATE_REELLE_DOC",
     "DATE_REELLE_DAACT"]]

BD TOPO

In [4]:
from credentials import s3
from BDTopo_fonctions import load_gpkg, gdf_DBSCAN, plot_DB_epsilon#, courbe_DB_epsilon
from BDTopo_fonctions import download_to_SSPCloud
from BDTopo_fonctions import upload_to_onyxia

gdf=load_gpkg("BDTOPO/BDTOPO_BATI_merge_dep_all_1000m3.gpkg")

Téléchargement depuis mgarbe/BDTOPO/BDTOPO_BATI_merge_dep_all_1000m3.gpkg ...
Chargement réussi (4120762 lignes)


In [4]:
import geopandas as gpd
import pandas as pd

def differentiel_BDTopo(gdf, seuil_aire=1000):
    
    if gdf.crs and gdf.crs.is_geographic:
        print("Attention: CRS géographique détecté. Reprojection recommandée.")
    
    gdf = gdf.copy()
    gdf['aire'] = gdf.geometry.area
    gdf = gdf.sort_values('Annee')
    annees = sorted(gdf['Annee'].unique())
    
    nb_nouveaux = 0
    nb_extensions = 0
    resultats = []

    for dep in sorted(gdf['Dep'].unique()):
        gdf_dep = gdf[gdf['Dep'] == dep].copy()
        nouveaux_batiments_dep = []

        for annee in annees:
            if annee < 2014:
                continue
            batiments_annee = gdf_dep[gdf_dep['Annee'] == annee].copy()
            batiments_precedents = gdf_dep[gdf_dep['Annee'] < annee].copy()

            nb_total = len(batiments_annee)
           

            if len(batiments_precedents) == 0:
                batiments_annee['Apparition_BDTopo'] = annee
                batiments_annee['type_nouveau'] = 'nouveau'
                nouveaux_batiments_dep.append(batiments_annee)
                continue

            for i, (idx, batiment) in enumerate(batiments_annee.iterrows(), 1):
                print(f"\r Traitement dep {dep}, année {annee}, jusqu'ici {nb_nouveaux} créations et {nb_extensions} extensions", 
                end="", flush=True)
                
                # Vérifier si l'ID existe dans les années précédentes
                batiment_meme_id = batiments_precedents[batiments_precedents['ID'] == batiment['ID']]
                
                # Vérifier les intersections spatiales
                intersections = batiments_precedents[
                    batiments_precedents.geometry.intersects(batiment.geometry)
                ]
                
                if len(batiment_meme_id) > 0 or len(intersections) > 0:
                    # Même ID ou intersection → vérifier extension
                    batiment_correspondant = None
                    if len(batiment_meme_id) > 0:
                        batiment_correspondant = batiment_meme_id.iloc[0]
                    elif len(intersections) > 0:
                        max_intersection_area = 0
                        for idx_prev, bat_prev in intersections.iterrows():
                            try:
                                intersection = batiment.geometry.intersection(bat_prev.geometry)
                                if intersection.is_empty:
                                    continue
                                intersection_area = intersection.area
                                if intersection_area > max_intersection_area:
                                    max_intersection_area = intersection_area
                                    batiment_correspondant = bat_prev
                            except Exception as e:
                                print(f"Erreur intersection pour ID {batiment['ID']}: {e}")
                                continue

                    if batiment_correspondant is not None:
                        diff_aire = batiment['aire'] - batiment_correspondant['aire']
                        if diff_aire > seuil_aire:
                            try:
                                nouvelle_geom = batiment.geometry.difference(batiment_correspondant.geometry)
                                if not nouvelle_geom.is_empty and nouvelle_geom.area > seuil_aire:
                                    nouveau_bat = batiment.copy()
                                    nouveau_bat.geometry = nouvelle_geom
                                    nouveau_bat['aire'] = nouvelle_geom.area
                                    nouveau_bat['type_nouveau'] = 'extension'
                                    nouveau_bat['Apparition_BDTopo'] = annee
                                    nouveaux_batiments_dep.append(pd.DataFrame([nouveau_bat]))
                                    nb_extensions += 1
                            except Exception as e:
                                print(f"Erreur calcul extension pour ID {batiment['ID']}: {e}")
                    else:
                        # intersection détectée mais aucune correspondance → nouveau
                        nouveau_bat = batiment.copy()
                        nouveau_bat['type_nouveau'] = 'nouveau'
                        nouveau_bat['Apparition_BDTopo'] = annee
                        nouveaux_batiments_dep.append(pd.DataFrame([nouveau_bat]))
                        nb_nouveaux += 1
                else:
                    # ID différent et pas d'intersection → nouveau bâtiment
                    nouveau_bat = batiment.copy()
                    nouveau_bat['type_nouveau'] = 'nouveau'
                    nouveau_bat['Apparition_BDTopo'] = annee
                    nouveaux_batiments_dep.append(pd.DataFrame([nouveau_bat]))
                    nb_nouveaux += 1

        if nouveaux_batiments_dep:
            resultats.append(pd.concat(nouveaux_batiments_dep, ignore_index=True))

    if resultats:
        final = pd.concat(resultats, ignore_index=True)
        final = gpd.GeoDataFrame(final, crs=gdf.crs)
        final['Apparition_BDTopo'] = final['Apparition_BDTopo'].astype(int)
        return final
    else:
        return gpd.GeoDataFrame(columns=list(gdf.columns) + ['Apparition_BDTopo', 'type_nouveau'], crs=gdf.crs)


In [5]:
import geopandas as gpd
import pandas as pd

def nouveaux_batiments(gdf, annee_min=2014, rayon=10):
    """
    Retourne un GeoDataFrame avec les bâtiments nouveaux à partir de annee_min
    en comparant ID et géométrie cumulée, par département.
    """
    gdf = gdf.copy()
    gdf['aire'] = gdf.geometry.area
    gdf = gdf.sort_values('Annee')
    annees = sorted(gdf['Annee'].unique())

    resultats = []

    nb_nouveaux = 0

    for dep in sorted(gdf['Dep'].unique()):
        gdf_dep = gdf[gdf['Dep'] == dep].copy()
        gdf_cumul = gdf_dep[gdf_dep['Annee'] < annee_min].copy()
        if not gdf_cumul.empty:
            sindex_cumul = gdf_cumul.sindex

        for annee in annees:
            if annee < annee_min:
                continue
            
            gdf_annee = gdf_dep[gdf_dep['Annee'] == annee].copy()
            nouveaux_idx = []
            nb_total = len(gdf_annee)
            

            for i, (idx, bat) in enumerate(gdf_annee.iterrows(), 1):
                print(f"\rTraitement du dep {dep}, année {annee} ; jusqu'ici {nb_nouveaux} nouveaux bâtiments detectés", end="", flush=True)
                # Vérification ID
                if bat['ID'] in gdf_cumul['ID'].values:
                    continue

                # Vérification position
                if not gdf_cumul.empty:
                    candidates = list(sindex_cumul.intersection(bat.geometry.buffer(rayon).bounds))
                    possible_matches = gdf_cumul.iloc[candidates]
                    if any(possible_matches.distance(bat.geometry) <= rayon):
                        continue

                # Nouveau bâtiment
                nb_nouveaux += 1
                nouveaux_idx.append(idx)

            gdf_annee_nouveaux = gdf_annee.loc[nouveaux_idx].copy()
            gdf_annee_nouveaux['Apparition_BDTopo'] = annee
            resultats.append(gdf_annee_nouveaux)

            # Mettre à jour cumul pour l'année suivante
            gdf_cumul = pd.concat([gdf_cumul, gdf_annee_nouveaux], ignore_index=True)
            if not gdf_cumul.empty:
                sindex_cumul = gdf_cumul.sindex

    if resultats:
        return gpd.GeoDataFrame(pd.concat(resultats, ignore_index=True), crs=gdf.crs)
    else:
        return gpd.GeoDataFrame(columns=gdf.columns.tolist() + ['Apparition_BDTopo'], crs=gdf.crs)



In [3]:
temp=nouveaux_batiments(gdf)

NameError: name 'nouveaux_batiments' is not defined

In [2]:
temp

NameError: name 'temp' is not defined